In [ ]:
import numpy as np
import pandas as pd
import sys
import os
from typing import List, Optional

if "Desktop" in os.getcwd():
    base_path = "."
else:
    base_path = "."

print(base_path)
sys.path.append(base_path)

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from src.utils.path_utils import get_project_root
from src.preprocessing.consts import Datasets
from src.preprocessing.read_raw_data import read_raw_data
from tabulate import tabulate

from src.preprocessing.consts import Datasets
from src.preprocessing.read_raw_data import read_raw_data, save_to_csv, read_csv, read_model_feature_injected_csv, read_weak_labeled_csv

pd.set_option("display.max_columns", None)

In [ ]:
df = read_weak_labeled_csv(Datasets.NELA_ELECTIONS_2020.weak_labeled_path, train_df_name=Datasets.NELA_ELECTIONS_SUBSET_2020.name)
df = df[["title", "content", "snorkel_majority_vote_label"]].copy()
df = df.rename(columns={"snorkel_majority_vote_label": "label", "content": "text"})

In [ ]:
df.head()

In [ ]:
df = df[["title", "content", "snorkel_majority_vote_label"]].copy()
df = df.rename(columns={"snorkel_majority_vote_label": "label", "content": "text"})

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df, df.snorkel_majority_vote_label.values, test_size = 0.2, random_state = 10)

In [ ]:
from transformers import AutoTokenizer

In [ ]:
TOKENIZER = AutoTokenizer.from_pretrained('bert-base-uncased')

In [ ]:
def tokenize(X):
    X = TOKENIZER(
        text = list(X),
        add_special_tokens = True,
        max_length = 100,
        truncation = True,
        padding = 'max_length',
        return_tensors = 'tf',
        return_token_type_ids = False,
        return_attention_mask = True,
        verbose = True
    )
    return X

In [ ]:
X_train_tokens = tokenize(X_train)
X_test_tokens = tokenize(X_test)

In [ ]:
import tensorflow as tf
from keras.models import Model, Sequential
from keras.layers import Input, Dense, Dropout, Embedding
from tensorflow.keras.optimizers import Adam
from transformers import TFBertModel

In [ ]:
Length = 100
def get_model():
    dropout_rate = 0.2
    input_ids = Input(shape = (Length,), dtype = tf.int32, name = 'input_ids')
    input_mask = Input(shape = (Length,), dtype = tf.int32, name = 'input_mask')
    embeddings = bert([input_ids, input_mask])[1] #pooler output
    print(embeddings)
    out = Dropout(0.2)(embeddings)
    #64 units dense layer
    out = Dense(64,activation = 'relu')(out)
    out = Dropout(0.2)(out)
    y = Dense(1,activation = 'sigmoid')(out)
    model = Model(inputs=[input_ids, input_mask], outputs=y)
    model.layers[2].trainable = True
    # define optimizer
    optimizer = Adam(learning_rate=1e-05, epsilon=1e-08, decay=0.01,clipnorm=1.0)
    #complile the model
    model.compile(optimizer = optimizer, loss = 'binary_crossentropy', metrics = 'accuracy')
    return model


In [ ]:
bert = TFBertModel.from_pretrained('bert-base-uncased')

In [ ]:
model = get_model()
tf.keras.utils.plot_model(model)

In [ ]:
from keras.callbacks import EarlyStopping

In [ ]:
history = model.fit(
    x={
        'input_ids':X_train_tokens['input_ids'],
        'input_mask':X_train_tokens['attention_mask'],
    },
    y=y_train,
    epochs=3,
    validation_split = 0.2,
    batch_size = 64,
    callbacks=[
        EarlyStopping(
            monitor='val_accuracy',
            mode='max',
            patience=3,
            verbose=False,
            restore_best_weights=True,
        )
    ]
)

In [ ]:
yhat = model.predict(
    {
        'input_ids' : X_test_tokens['input_ids'] ,
        'input_mask' : X_test_tokens['attention_mask']
    }
)

In [ ]:
X_train_tokens

In [ ]:
print(classification_report(y_test,yhat))

In [ ]:
yhat